In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
data = pd.read_csv("../reviews_filtered.csv")

print(data.head(10))

   Unnamed: 0  movie_id                       name  count_  user_id  rating
0           0     34222  slumber-party-massacre-ii      32   438333       7
1           1     34222  slumber-party-massacre-ii      32   437213       8
2           2     34222  slumber-party-massacre-ii      32   434710       7
3           3     34222  slumber-party-massacre-ii      32   423063       5
4           4     34222  slumber-party-massacre-ii      32   244269       9
5           5     34222  slumber-party-massacre-ii      32   188272       7
6           6     34222  slumber-party-massacre-ii      32    30514       6
7           7     34222  slumber-party-massacre-ii      32     8026      10
8           8    462065         run-sweetheart-run      32  1146451       5
9           9    462065         run-sweetheart-run      32  1067317       7


import re

love_death_robots_filter = re.compile("love-death-robots")
black_mirror_filter = re.compile("black-mirror")

to_be_removed = []

for idx, row in data.iterrows():
    if (re.match(love_death_robots_filter, row['name'])):
        to_be_removed.append(idx)
    elif (re.match(black_mirror_filter, row['name'])):
        to_be_removed.append(idx)

data = data.drop(to_be_removed)

data.to_csv("../reviews_filtered.csv")

In [3]:
unique_movies = sorted(list(set(zip(data['movie_id'], data['name']))))
unique_users = sorted(list(set(data['user_id'])))

num_movies = len(unique_movies)
num_users = len(unique_users)

print("%d Unique Movies" % num_movies)
print("%d Unique Users" % num_users)

5937 Unique Movies
2516 Unique Users


In [4]:
movie_id_to_idx = {movie_id: idx for idx,
                   (movie_id, _) in enumerate(unique_movies)}
user_id_to_idx = {user_id: idx for idx, user_id in enumerate(unique_users)}

movie_name_to_idx = {movie_name: idx for idx,
                     (_, movie_name) in enumerate(unique_movies)}

movie_names = [movie_name for (_, movie_name) in unique_movies]

movie_id_to_name = {movie_id: movie_name for (
    movie_id, movie_name) in unique_movies}

movie_name_to_idx = {
    movie_id_to_name[movie_id]: idx for idx, (movie_id, _) in enumerate(unique_movies)}

movie_freqs = {movie_id_to_idx[movie_id]: freq for (
    movie_id, freq) in zip(data['movie_id'], data['count_'])}

In [5]:
def look_up_movie(movie_name):
    match_string = re.compile(".*" + movie_name + ".*")
    for idx, (movie_id, movie_name) in enumerate(unique_movies):
        # movie_name = movie_id_to_name[movie_id]
        if (re.match(match_string, movie_name)):
            yield (movie_name, movie_id, idx)

In [6]:
for (movie_name, movie_id, idx) in look_up_movie("emma"):
    print(movie_name, movie_id, idx)

emma-2020 485212 4103
the-social-dilemma 581924 4541


In [7]:
from sklearn.model_selection import train_test_split
from surprise import Dataset, NormalPredictor, Reader


reader = Reader(rating_scale=(1, 10))

dataset = Dataset.load_from_df(data[["user_id", "movie_id", "rating"]], reader)

In [8]:
from surprise import accuracy, Dataset, SVD
from surprise.model_selection import KFold

# define a cross-validation iterator
kf = KFold(n_splits=3)

algo = SVD()

for trainset, testset in kf.split(dataset):

    # train and test algorithm.
    algo.fit(trainset)
    predictions = algo.test(testset)

    # Compute and print Root Mean Squared Error
    accuracy.rmse(predictions, verbose=True)

RMSE: 1.5372
RMSE: 1.5423
RMSE: 1.5401


In [45]:
my_ratings = {"cars": 10, "cars-2": 10, "cars-3": 10}

my_ratings_set = data.copy(deep=True)

user_id = num_users

for movie_name, rating in my_ratings.items():
    if(movie_name not in movie_names):
        continue
    movie_idx = movie_name_to_idx[movie_name]

    my_ratings_set.loc[-1] = [0, movie_idx, movie_name, 0, user_id, rating]  # adding a row
    my_ratings_set.index = my_ratings_set.index + 1  # shifting index
    my_ratings_set = my_ratings_set.sort_index()  # sorting by index

recommender_set = Dataset.load_from_df(
    data[["user_id", "movie_id", "rating"]], reader).build_full_trainset()

In [47]:
from surprise.dataset import DatasetAutoFolds

test_set = recommender_set.build_anti_testset()

print(len(test_set))
print(len(test_set[0]))
print(test_set[0])

14082058
3
(438333, 462065, 6.630305786302626)


In [48]:
from collections import defaultdict

from surprise import Dataset, SVD


def get_top_n(predictions, n=10):
    """Return the top-N recommendation for each user from a set of predictions.

    Args:
        predictions(list of Prediction objects): The list of predictions, as
            returned by the test method of an algorithm.
        n(int): The number of recommendation to output for each user. Default
            is 10.

    Returns:
    A dict where keys are user (raw) ids and values are lists of tuples:
        [(raw item id, rating estimation), ...] of size n.
    """

    # First map the predictions to each user.
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))

    # Then sort the predictions for each user and retrieve the k highest ones.
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]

    return top_n

In [49]:
predictions = algo.test(test_set)

print(len(predictions))

top_n = get_top_n(predictions, n=10)

KeyboardInterrupt: 

In [27]:
for uid, user_ratings in top_n.items():
    print(uid, [iid for (iid, _) in user_ratings])